In [1]:
import json
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
def split_pdf(file_path: str):

    elements = partition_pdf(
        filename= file_path,
        extract_image_block_to_payload= True,
        infer_table_structure=True,
        extract_images_in_pdf= True,
        extract_image_block_types= ["Images"]
    )

    print(f"Extracted {len(elements)} elements")
    return elements

In [4]:
def split_by_title(element):

    chunks = chunk_by_title(
        elements= element,
        max_characters= 3000,
        new_after_n_chars= 2400,
        combine_text_under_n_chars= 500 
    )

    print(f"Extracted {len(chunks)} chunks")
    return chunks

In [5]:
def separate_content(chunk):

    content_data = {
        "Text": chunk.text,
        "Types": ["text"],
        "Tables" : [],
        "Images": []
    }

    if hasattr(chunk, "metadata") and hasattr(chunk.metadata, "orig_elements"):
        for el in chunk.metadata.orig_elements:
            if type(el).__name__ == "Table":
                table_html = getattr(el.metadata, "text_as_html", el.text)
                content_data["Tables"].append(table_html)
                content_data["Types"].append("table")

            if type(el).__name__ == "Image":
                if hasattr(el, 'metadata') and hasattr(el.metadata, 'image_base64'):
                    content_data["Types"].append("image")
                    img_64 = getattr(el.metadata, "image_base64", el.text)
                    content_data["Images"].append(img_64)

        content_data["Types"] = List(set(content_data["Types"]))

        return content_data

In [10]:
openrouter_key = os.environ.get("OPENROUTER_API_KEY")
if not openrouter_key:

    raise ValueError("OPENROUTER_API_KEY is missing from environment variables.")
    # Configures OpenRouter base URL to access hosted open-source vision models
chat_model = ChatOpenAI(
    model="qwen/qwen2.5-vl-72b-instruct",
    openai_api_key=openrouter_key, # Insert your free/cheap OpenRouter API key
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.2, # inline comment: low temperature minimizes hallucination
    max_tokens=512 # inline comment: token limit per response payload
)

def create_ai_summary(text :str ,tables : List[str], images: List[str]):
    try:
        prompt = f"CONTENT TO ANALYZE:\nTEXT:\n{text}\n"

        if tables:
            prompt += "\nTables :\n"
            for _, table in enumerate(tables):
                prompt += "Table{_} -> {table}\n"

        prompt += "\nProvide a concise, searchable summary of the content above:"
        message = [
            {
                "type" : "text",
                "text" : prompt
            }
        ]

        if images:
            for img in images:
                message.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{img}"}
                })

        response = chat_model.invoke([HumanMessage(content=message)])
        return response.content
    except Exception as e:
        print(f"Summary Failed : {e}")
        return text[:300]

In [17]:
def summarize_chunks(chunkes):
    print("🧠 Processing chunks with AI Summaries...")
        
    langchain_documents = []
    total_chunks = len(chunkes)
    
    for i, chunk in enumerate(chunkes):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        content_data = separate_content(chunk)
        
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            enhanced_content = content_data['text']
        
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents

In [18]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5" # Free open-source state-of-the-art embedding model
)

def create_vec_Store(docs : List[Document], directory = r"D:\Course\Projects\00. Vectory DataBases\1. Attention"):

    vecStore = Chroma.from_documents(
        documents= docs,
        embedding= embedding_model,
        collection_metadata={"hnsw:space": "cosine"}
    )

    print("Vector Store Created")
    return vecStore

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-VL-72B-Instruct", # Large multimodal open-source model
    temperature=0.1, # low temperature ensures high factual accuracy
    max_new_tokens=1024, # token limit for comprehensive response output
    timeout=60.0 # handle larger model request overhead safely
)
vision_llm = ChatHuggingFace(llm=llm)
def generate_final_answer(chunks, query):

    try:
        prompt_text = f"""Based on the following documents, please answer this question: {query}\n\nCONTENT TO ANALYZE:\n"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """Please provide a clear answer. If documents lack info, say "I don't have enough information."\n\nANSWER:"""

        message_content = [{"type": "text", "text": prompt_text}]
        
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        message = HumanMessage(content=message_content)
        response = vision_llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

In [20]:
def run_complete_ingestion_pipeline(pdf_path: str):
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    elements = split_pdf(pdf_path)
    chunks = split_by_title(elements)
    summarised_chunks = summarize_chunks(chunks)
    db = create_vec_Store(summarised_chunks, persist_directory="dbv2/chroma_db")
    
    print("🎉 Pipeline completed successfully!")
    return db

# Execution Flow
db = run_complete_ingestion_pipeline(r"D:\Course\Projects\0. Documents\Attention.pdf")

🚀 Starting RAG Ingestion Pipeline


No languages specified, defaulting to English.
The requested type (Images) doesn't match any available type


Extracted 266 elements
Extracted 33 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/33


TypeError: Type List cannot be instantiated; use list() instead